In [27]:
import pathlib as pl
import pandas as pd
import numpy as np

In [28]:
root_path = pl.Path.cwd().parent.parent
data_path = root_path / "data" / "intermed" 

In [29]:
data_review = pd.read_csv(data_path / "googleplaystore_user_reviews_clean.csv")
data_apps = pd.read_csv(data_path / "googleplaystore_clean.csv")

In [ ]:
df_sentimento = pd.merge(data_apps[['App', 'Category']], data_review[['App', 'Sentiment_Polarity']], on='App', how='inner')

genre_sentiment = df_sentimento.groupby('Category').agg(
    Sent_Mean=('Sentiment_Polarity', 'mean'),
    Sent_Std=('Sentiment_Polarity', 'std')
).reset_index()

genre_sentiment['Sent_Std'] = genre_sentiment['Sent_Std'].fillna(0)

grouped = data_apps.groupby('Category').agg(
    C=('App', 'count'),              
    A=('Rating', 'mean'),            
    D=('Installs', 'mean'),           
    U=('Days_Since_Update', 'mean')  
).reset_index()

grouped = grouped[grouped['Category'] != '1.9']

grouped = pd.merge(grouped, genre_sentiment, on='Category', how='left')
grouped['Sent_Mean'] = grouped['Sent_Mean'].fillna(0)
grouped['Sent_Std'] = grouped['Sent_Std'].fillna(0)

gp_data = data_apps.groupby("Category").agg({
    "Rating": "mean", 
    "Installs": "mean",
    "App":"count"
}).reset_index()
limite_downloads = gp_data['Installs'].quantile(0.70) 
grouped = grouped[grouped['D'] >= limite_downloads].copy()


grouped = grouped.reset_index(drop=True)

grouped['U_norm'] = grouped['U'] / grouped['U'].max()
grouped['C_norm'] = 1 - (grouped['C'] / grouped['C'].max())
grouped['A_norm'] = 1 - (grouped['A'] / 5.0)
grouped['D_log'] = np.log10(grouped['D'] + 1)
grouped['D_norm'] = grouped['D_log'] / grouped['D_log'].max()

grouped['Score_Oportunidade'] = grouped['Sent_Std'] - grouped['Sent_Mean']

min_score = grouped['Score_Oportunidade'].min()
max_score = grouped['Score_Oportunidade'].max()
grouped['P_norm'] = (grouped['Score_Oportunidade'] - min_score) / (max_score - min_score)


In [31]:
grouped.head()

,Category,C,A,D,U,Sent_Mean,Sent_Std,U_norm,C_norm,A_norm,D_log,D_norm,Score_Oportunidade,P_norm
0,COMMUNICATION,307,4.151466,7.867180e+07,248.175896,0.175969,0.341416,0.696306,0.717831,0.169707,7.895819,1.000000,0.165447,0.348477
1,ENTERTAINMENT,121,4.142975,2.099388e+07,82.553719,0.135963,0.341373,0.231621,0.888787,0.171405,7.322093,0.927338,0.205410,0.657271
2,GAME,1088,4.283180,2.964059e+07,279.249081,0.061846,0.270508,0.783488,0.000000,0.143364,7.471887,0.946309,0.208662,0.682402
3,NEWS_AND_MAGAZINES,214,4.128505,2.520145e+07,153.560748,0.150623,0.317243,0.430845,0.803309,0.174299,7.401426,0.937385,0.166621,0.357544
4,PHOTOGRAPHY,304,4.182895,3.197777e+07,242.763158,0.222870,0.381987,0.681120,0.720588,0.163421,7.504848,0.950484,0.159116,0.299559


In [32]:
grouped.to_csv(root_path / "data" / "processed" / "googleplaystore_normalized.csv", index=False)

In [ ]:
merged_data = pd.merge(left=data_apps,right=data_review,how="inner",on=["App"])